# FinanceAI — Módulo de Perfil Financiero

Notebook con el flujo completo del módulo **Perfil Financiero** (Saludable / En observación / En riesgo):

1. Definición de columnas y criterios
2. Función única de reglas (fuente de verdad, sin duplicación)
3. Generación del dataset sintético
4. Casos de prueba curados a mano (casos límite)
5. Validación automática de las reglas
6. Explicabilidad (el "porqué" de cada veredicto)
7. Wrapper listo para conectar con el endpoint `/analisis-financiero`
8. (Opcional) Entrenamiento de un modelo con el dataset sintético

**Marco de referencia usado para calibrar los umbrales:**
- Financial Health Network — *FinHealth Score* (estructura de 3 tiers: Healthy / Coping / Vulnerable, y 4 pilares: Spend, Save, Borrow, Plan)
- Fannie Mae *Selling Guide B3-6-02* (umbrales de Debt-to-Income: 36% / 43%)
- Regla de presupuesto 50/30/20 (umbral de ahorro saludable ≈ 20%)


In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)
np.random.seed(42)


## 1. Columnas del dataset y criterios de negocio

**Columnas del dataset de perfiles**:

| Columna | Tipo | Descripción |
|---|---|---|
| `ingreso_mensual` | numérico | Ingreso mensual del usuario |
| `nivel_endeudamiento` | numérico (%) | % del ingreso comprometido en deudas |
| `frecuencia_ahorro` | categórico | Baja / Media / Alta (informativo, no decide el veredicto) |
| `gasto_total_mes` | numérico | Suma de las transacciones del mes (viene del módulo de clasificación de gastos) |
| `ratio_gasto_ingreso` | numérico (derivado) | `gasto_total_mes / ingreso_mensual` — variable central de la regla |
| `perfil` | categórico (target) | Saludable / En observación / En riesgo |

**Criterios (umbrales validados contra fuentes reales de la industria financiera):**

| Nivel de endeudamiento | Categoría |
|---|---|
| ≤ 36% | Saludable |
| 36% – 43% | En observación |
| > 43% | En riesgo |

| ratio_gasto_ingreso | Categoría |
|---|---|
| ≤ 0.80 | Saludable |
| 0.80 – 0.90 | En observación |
| > 0.90 | En riesgo |

Se usa el **peor caso entre ambos criterios** (enfoque conservador, igual que en underwriting bancario): si cualquiera de los dos indica riesgo u observación, el perfil sube a esa categoría.

`frecuencia_ahorro` se mantiene como columna descriptiva en la respuesta (útil para el dashboard y las recomendaciones), pero **no participa en la lógica de decisión** porque es prácticamente complementaria de `ratio_gasto_ingreso` (ingreso ≈ gasto + ahorro), y usar ambas duplicaría la misma señal.


## 2. Función de reglas — fuente única de verdad

Esta es la **única** función que define la lógica de negocio. Tanto el generador del dataset sintético como la API la usan (evita tener el mismo criterio escrito en dos lugares que se puedan desincronizar).


In [2]:
def calcular_perfil(nivel_endeudamiento: float, ratio_gasto_ingreso: float):
    """
    Fuente unica de verdad para la logica de perfil financiero.
    Devuelve (perfil, razones).
    """
    razones = []

    # --- En riesgo ---
    if nivel_endeudamiento > 43:
        razones.append("el nivel de endeudamiento supera el 43% del ingreso")
    if ratio_gasto_ingreso > 0.9:
        razones.append("los gastos representan mas del 90% del ingreso mensual")

    if razones:
        return "En riesgo", razones

    # --- En observacion ---
    if 36 <= nivel_endeudamiento <= 43:
        razones.append("el endeudamiento esta en zona moderada (36%-43%)")
    if 0.8 <= ratio_gasto_ingreso <= 0.9:
        razones.append("los gastos representan entre el 80% y 90% del ingreso")

    if razones:
        return "En observacion", razones

    # --- Saludable ---
    return "Saludable", ["endeudamiento controlado y gasto razonable frente al ingreso"]


In [3]:
def analizar_perfil(ingreso_mensual: float, nivel_endeudamiento: float,
                     frecuencia_ahorro: str, gasto_total_mes: float) -> dict:
    """
    Wrapper listo para el endpoint /analisis-financiero.
    Calcula el ratio, llama a calcular_perfil() y arma la respuesta en el
    formato JSON que espera el reto.
    """
    ratio = round(gasto_total_mes / ingreso_mensual, 2)
    perfil, razones = calcular_perfil(nivel_endeudamiento, ratio)

    return {
        "perfil_financiero": perfil,
        "razones": razones,
        "metricas": {
            "ratio_gasto_ingreso": ratio,
            "nivel_endeudamiento": nivel_endeudamiento,
            "frecuencia_ahorro": frecuencia_ahorro
        }
    }

# Prueba rapida
analizar_perfil(ingreso_mensual=4500, nivel_endeudamiento=25,
                 frecuencia_ahorro="Media", gasto_total_mes=3825)


{'perfil_financiero': 'En observacion',
 'razones': ['los gastos representan entre el 80% y 90% del ingreso'],
 'metricas': {'ratio_gasto_ingreso': 0.85,
  'nivel_endeudamiento': 25,
  'frecuencia_ahorro': 'Media'}}

## 3. Dataset sintético (para EDA y para entrenar un modelo, si el tiempo alcanza)

Se genera con distribuciones aleatorias y se etiqueta usando la **misma** `calcular_perfil()` — así el dataset queda perfectamente consistente con la lógica de reglas.


In [4]:
N = 500

df = pd.DataFrame({
    "ingreso_mensual": np.random.uniform(300_000, 3_000_000, N).round(0),
    "nivel_endeudamiento": np.random.uniform(0, 70, N).round(1),
    "frecuencia_ahorro": np.random.choice(["Baja", "Media", "Alta"], N, p=[0.4, 0.4, 0.2]),
})

# gasto total como proporcion variable del ingreso, con ruido
df["gasto_total_mes"] = (df["ingreso_mensual"] * np.random.uniform(0.3, 1.1, N)).round(0)
df["ratio_gasto_ingreso"] = (df["gasto_total_mes"] / df["ingreso_mensual"]).round(2)

def etiquetar_fila(row):
    perfil, _ = calcular_perfil(row["nivel_endeudamiento"], row["ratio_gasto_ingreso"])
    return perfil

df["perfil"] = df.apply(etiquetar_fila, axis=1)

print(df["perfil"].value_counts())
df.head()


perfil
En riesgo         247
Saludable         171
En observacion     82
Name: count, dtype: int64


,ingreso_mensual,nivel_endeudamiento,frecuencia_ahorro,gasto_total_mes,ratio_gasto_ingreso,perfil
0,1311258.0,48.9,Baja,937898.0,0.72,En riesgo
1,2866929.0,37.5,Media,1959103.0,0.68,En observacion
2,2276384.0,21.7,Alta,729612.0,0.32,Saludable
3,1916378.0,57.0,Media,1098081.0,0.57,En riesgo
4,721250.0,47.9,Alta,435748.0,0.60,En riesgo


In [5]:
# EDA rapido
df.describe(include="all")


,ingreso_mensual,nivel_endeudamiento,frecuencia_ahorro,gasto_total_mes,ratio_gasto_ingreso,perfil
count,5.000000e+02,500.00000,500,5.000000e+02,500.000000,500
unique,NaN,NaN,3,NaN,NaN,3
top,NaN,NaN,Media,NaN,NaN,En riesgo
freq,NaN,NaN,200,NaN,NaN,247
mean,1.646117e+06,33.73760,NaN,1.144951e+06,0.697140,NaN
std,8.064587e+05,19.98739,NaN,7.105642e+05,0.229523,NaN
min,3.136660e+05,0.30000,NaN,1.062860e+05,0.300000,NaN
25%,9.514550e+05,16.00000,NaN,5.691725e+05,0.490000,NaN
50%,1.685542e+06,33.00000,NaN,1.009902e+06,0.710000,NaN
75%,2.341537e+06,50.85000,NaN,1.546247e+06,0.890000,NaN


In [6]:
df.to_csv("dataset_perfil_financiero.csv", index=False)
print("Guardado: dataset_perfil_financiero.csv")


Guardado: dataset_perfil_financiero.csv


## 4. Casos de prueba curados a mano (casos límite)

A diferencia del dataset sintético (aleatorio), estos casos se definieron **a mano**, con el resultado esperado decidido *antes* de correr el código. Sirven para validar que la función de reglas se comporta como se espera, especialmente en los bordes exactos (36%, 43%, 0.80, 0.90).


In [7]:
casos_prueba = pd.DataFrame([
    {"caso": "Saludable claro",                    "ingreso_mensual": 1_200_000, "nivel_endeudamiento": 10,   "frecuencia_ahorro": "Alta",  "gasto_total_mes": 800_000,  "perfil_esperado": "Saludable"},
    {"caso": "Riesgo por endeudamiento alto",       "ingreso_mensual": 1_200_000, "nivel_endeudamiento": 50,   "frecuencia_ahorro": "Baja",  "gasto_total_mes": 700_000,  "perfil_esperado": "En riesgo"},
    {"caso": "Riesgo por gasto excesivo",           "ingreso_mensual": 1_200_000, "nivel_endeudamiento": 15,   "frecuencia_ahorro": "Baja",  "gasto_total_mes": 1_150_000,"perfil_esperado": "En riesgo"},
    {"caso": "Observacion por endeudamiento",       "ingreso_mensual": 1_200_000, "nivel_endeudamiento": 38,   "frecuencia_ahorro": "Media", "gasto_total_mes": 800_000,  "perfil_esperado": "En observacion"},
    {"caso": "Observacion por ratio de gasto",      "ingreso_mensual": 1_200_000, "nivel_endeudamiento": 10,   "frecuencia_ahorro": "Media", "gasto_total_mes": 1_020_000,"perfil_esperado": "En observacion"},

    # Casos limite exactos
    {"caso": "Limite exacto 36% deuda",             "ingreso_mensual": 1_000_000, "nivel_endeudamiento": 36,   "frecuencia_ahorro": "Media", "gasto_total_mes": 700_000,  "perfil_esperado": "En observacion"},
    {"caso": "Justo bajo el limite (35.9%)",        "ingreso_mensual": 1_000_000, "nivel_endeudamiento": 35.9, "frecuencia_ahorro": "Alta",  "gasto_total_mes": 700_000,  "perfil_esperado": "Saludable"},
    {"caso": "Limite exacto 43% deuda",             "ingreso_mensual": 1_000_000, "nivel_endeudamiento": 43,   "frecuencia_ahorro": "Media", "gasto_total_mes": 700_000,  "perfil_esperado": "En observacion"},
    {"caso": "Justo sobre el limite (43.1%)",       "ingreso_mensual": 1_000_000, "nivel_endeudamiento": 43.1, "frecuencia_ahorro": "Baja",  "gasto_total_mes": 700_000,  "perfil_esperado": "En riesgo"},
    {"caso": "Ratio exacto 0.80",                   "ingreso_mensual": 1_000_000, "nivel_endeudamiento": 10,   "frecuencia_ahorro": "Media", "gasto_total_mes": 800_000,  "perfil_esperado": "En observacion"},
    {"caso": "Ratio exacto 0.90",                   "ingreso_mensual": 1_000_000, "nivel_endeudamiento": 10,   "frecuencia_ahorro": "Baja",  "gasto_total_mes": 900_000,  "perfil_esperado": "En observacion"},
    {"caso": "Ratio justo sobre 0.90 (0.91)",       "ingreso_mensual": 1_000_000, "nivel_endeudamiento": 10,   "frecuencia_ahorro": "Baja",  "gasto_total_mes": 910_000,  "perfil_esperado": "En riesgo"},

    # Casos contradictorios
    {"caso": "Deuda baja pero gasta casi todo",     "ingreso_mensual": 900_000,   "nivel_endeudamiento": 5,    "frecuencia_ahorro": "Baja",  "gasto_total_mes": 880_000,  "perfil_esperado": "En riesgo"},
    {"caso": "Deuda alta pero gasta poco",          "ingreso_mensual": 900_000,   "nivel_endeudamiento": 45,   "frecuencia_ahorro": "Alta",  "gasto_total_mes": 400_000,  "perfil_esperado": "En riesgo"},

    # Extremos de ingreso
    {"caso": "Ingreso muy bajo, bien administrado", "ingreso_mensual": 350_000,   "nivel_endeudamiento": 8,    "frecuencia_ahorro": "Alta",  "gasto_total_mes": 200_000,  "perfil_esperado": "Saludable"},
    {"caso": "Ingreso alto, mal administrado",      "ingreso_mensual": 5_000_000, "nivel_endeudamiento": 48,   "frecuencia_ahorro": "Baja",  "gasto_total_mes": 4_800_000,"perfil_esperado": "En riesgo"},
])

casos_prueba["ratio_gasto_ingreso"] = (casos_prueba["gasto_total_mes"] / casos_prueba["ingreso_mensual"]).round(2)
casos_prueba.to_csv("casos_prueba_perfil.csv", index=False)
casos_prueba


,caso,ingreso_mensual,nivel_endeudamiento,frecuencia_ahorro,gasto_total_mes,perfil_esperado,ratio_gasto_ingreso
0,Saludable claro,1200000,10.0,Alta,800000,Saludable,0.67
1,Riesgo por endeudamiento alto,1200000,50.0,Baja,700000,En riesgo,0.58
2,Riesgo por gasto excesivo,1200000,15.0,Baja,1150000,En riesgo,0.96
3,Observacion por endeudamiento,1200000,38.0,Media,800000,En observacion,0.67
4,Observacion por ratio de gasto,1200000,10.0,Media,1020000,En observacion,0.85
5,Limite exacto 36% deuda,1000000,36.0,Media,700000,En observacion,0.70
6,Justo bajo el limite (35.9%),1000000,35.9,Alta,700000,Saludable,0.70
7,Limite exacto 43% deuda,1000000,43.0,Media,700000,En observacion,0.70
8,Justo sobre el limite (43.1%),1000000,43.1,Baja,700000,En riesgo,0.70
9,Ratio exacto 0.80,1000000,10.0,Media,800000,En observacion,0.80


## 5. Validación automática de las reglas

In [8]:
def validar_reglas(casos_df: pd.DataFrame) -> pd.DataFrame:
    resultados = []
    for _, row in casos_df.iterrows():
        resultado = analizar_perfil(
            row["ingreso_mensual"], row["nivel_endeudamiento"],
            row["frecuencia_ahorro"], row["gasto_total_mes"]
        )
        ok = resultado["perfil_financiero"] == row["perfil_esperado"]
        resultados.append({
            "caso": row["caso"],
            "esperado": row["perfil_esperado"],
            "obtenido": resultado["perfil_financiero"],
            "correcto": ok,
            "razones": "; ".join(resultado["razones"])
        })
    return pd.DataFrame(resultados)

reporte = validar_reglas(casos_prueba)
display(reporte)

aciertos = reporte["correcto"].sum()
total = len(reporte)
print(f"\nAciertos: {aciertos}/{total}")

if aciertos < total:
    print("\n⚠ Revisar los siguientes casos fallidos:")
    display(reporte[~reporte["correcto"]])
else:
    print("✅ Todas las reglas pasan la validacion.")


,caso,esperado,obtenido,correcto,razones
0,Saludable claro,Saludable,Saludable,True,endeudamiento controlado y gasto razonable fre...
1,Riesgo por endeudamiento alto,En riesgo,En riesgo,True,el nivel de endeudamiento supera el 43% del in...
2,Riesgo por gasto excesivo,En riesgo,En riesgo,True,los gastos representan mas del 90% del ingreso...
3,Observacion por endeudamiento,En observacion,En observacion,True,el endeudamiento esta en zona moderada (36%-43%)
4,Observacion por ratio de gasto,En observacion,En observacion,True,los gastos representan entre el 80% y 90% del ...
5,Limite exacto 36% deuda,En observacion,En observacion,True,el endeudamiento esta en zona moderada (36%-43%)
6,Justo bajo el limite (35.9%),Saludable,Saludable,True,endeudamiento controlado y gasto razonable fre...
7,Limite exacto 43% deuda,En observacion,En observacion,True,el endeudamiento esta en zona moderada (36%-43%)
8,Justo sobre el limite (43.1%),En riesgo,En riesgo,True,el nivel de endeudamiento supera el 43% del in...
9,Ratio exacto 0.80,En observacion,En observacion,True,los gastos representan entre el 80% y 90% del ...



Aciertos: 16/16
✅ Todas las reglas pasan la validacion.


## 6. Ejemplos reales de uso

Formato alineado al ejemplo `POST /analisis-financiero` del PDF del reto.


In [9]:
ejemplos = [
    dict(ingreso_mensual=4500, nivel_endeudamiento=25, frecuencia_ahorro="Media", gasto_total_mes=3825),
    dict(ingreso_mensual=2_000_000, nivel_endeudamiento=45, frecuencia_ahorro="Baja", gasto_total_mes=1_900_000),
    dict(ingreso_mensual=1_500_000, nivel_endeudamiento=15, frecuencia_ahorro="Alta", gasto_total_mes=900_000),
]

for i, ej in enumerate(ejemplos, 1):
    print(f"--- Ejemplo {i} ---")
    print("Entrada:", ej)
    print("Salida: ", analizar_perfil(**ej))
    print()


--- Ejemplo 1 ---
Entrada: {'ingreso_mensual': 4500, 'nivel_endeudamiento': 25, 'frecuencia_ahorro': 'Media', 'gasto_total_mes': 3825}
Salida:  {'perfil_financiero': 'En observacion', 'razones': ['los gastos representan entre el 80% y 90% del ingreso'], 'metricas': {'ratio_gasto_ingreso': 0.85, 'nivel_endeudamiento': 25, 'frecuencia_ahorro': 'Media'}}

--- Ejemplo 2 ---
Entrada: {'ingreso_mensual': 2000000, 'nivel_endeudamiento': 45, 'frecuencia_ahorro': 'Baja', 'gasto_total_mes': 1900000}
Salida:  {'perfil_financiero': 'En riesgo', 'razones': ['el nivel de endeudamiento supera el 43% del ingreso', 'los gastos representan mas del 90% del ingreso mensual'], 'metricas': {'ratio_gasto_ingreso': 0.95, 'nivel_endeudamiento': 45, 'frecuencia_ahorro': 'Baja'}}

--- Ejemplo 3 ---
Entrada: {'ingreso_mensual': 1500000, 'nivel_endeudamiento': 15, 'frecuencia_ahorro': 'Alta', 'gasto_total_mes': 900000}
Salida:  {'perfil_financiero': 'Saludable', 'razones': ['endeudamiento controlado y gasto razona

## 7. Modelo entrenado

Usa el dataset sintético del paso 3. El modelo da el veredicto + probabilidad; `calcular_perfil()` sigue dando el "porqué" (explicabilidad),


In [10]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import joblib

X = df[["nivel_endeudamiento", "ratio_gasto_ingreso"]]
y = df["perfil"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

modelo = RandomForestClassifier(n_estimators=100, random_state=42)
modelo.fit(X_train, y_train)

print(classification_report(y_test, modelo.predict(X_test)))


                precision    recall  f1-score   support

En observacion       1.00      1.00      1.00        16
     En riesgo       1.00      1.00      1.00        50
     Saludable       1.00      1.00      1.00        34

      accuracy                           1.00       100
     macro avg       1.00      1.00      1.00       100
  weighted avg       1.00      1.00      1.00       100



In [11]:
joblib.dump(modelo, "modelo_perfil_financiero.pkl")
print("Modelo guardado: modelo_perfil_financiero.pkl")


Modelo guardado: modelo_perfil_financiero.pkl


## 8. Resumen de artefactos generados

- `dataset_perfil_financiero.csv` — dataset sintético (500 filas) para EDA y entrenamiento
- `casos_prueba_perfil.csv` — 14 casos curados a mano para validación
- `modelo_perfil_financiero.pkl` — modelo entrenado (opcional)
- Funciones listas para el backend: `calcular_perfil()` (lógica pura) y `analizar_perfil()` (wrapper para el endpoint)

**Siguiente paso del checklist:** conectar `analizar_perfil()` al endpoint `POST /analisis-financiero` del backend (tarea de la sección 4 — API).


## 9. Dataset con ambigüedad realista (revisión)

El dataset del paso 3 etiquetaba con una frontera perfectamente limpia (misma fórmula usada para generar y para etiquetar), lo que hacía que cualquier modelo entrenado saliera con `probabilidad` casi siempre en 0.99-1.0 — poco realista y poco informativo.

Aquí regeneramos el dataset agregando **ambigüedad cerca de los bordes de decisión**: los casos que caen muy cerca de un umbral (36%, 43%, 0.80, 0.90) tienen una probabilidad de que la etiqueta observada se corra a la categoría vecina — simula la imprecisión normal de datos financieros reportados por usuarios reales. Esto crea zonas grises genuinas que el modelo debe aprender a manejar.


In [12]:
def calcular_perfil_reglas(nivel_endeudamiento, ratio_gasto_ingreso):
    if nivel_endeudamiento > 43 or ratio_gasto_ingreso > 0.9:
        return "En riesgo"
    elif (36 <= nivel_endeudamiento <= 43) or (0.8 <= ratio_gasto_ingreso <= 0.9):
        return "En observacion"
    else:
        return "Saludable"

def etiquetar_con_ambiguedad(row, margen_deuda=4, margen_ratio=0.04, prob_flip=0.35):
    """
    Etiqueta base = regla exacta. Si el caso cae MUY cerca de un borde,
    hay prob_flip de que la etiqueta observada se corra a la categoria vecina.
    """
    perfil = calcular_perfil_reglas(row["nivel_endeudamiento"], row["ratio_gasto_ingreso"])
    cerca_borde = (
        abs(row["nivel_endeudamiento"] - 36) < margen_deuda or
        abs(row["nivel_endeudamiento"] - 43) < margen_deuda or
        abs(row["ratio_gasto_ingreso"] - 0.8) < margen_ratio or
        abs(row["ratio_gasto_ingreso"] - 0.9) < margen_ratio
    )
    if cerca_borde and np.random.rand() < prob_flip:
        orden = ["Saludable", "En observacion", "En riesgo"]
        idx = orden.index(perfil)
        vecino = max(0, min(2, idx + np.random.choice([-1, 1])))
        return orden[vecino]
    return perfil

N = 800
df = pd.DataFrame({
    "ingreso_mensual": np.random.uniform(300_000, 3_000_000, N).round(0),
    "nivel_endeudamiento": np.random.uniform(0, 70, N).round(1),
    "frecuencia_ahorro": np.random.choice(["Baja", "Media", "Alta"], N, p=[0.4, 0.4, 0.2]),
})
df["gasto_total_mes"] = (df["ingreso_mensual"] * np.random.uniform(0.3, 1.1, N)).round(0)
df["ratio_gasto_ingreso"] = (df["gasto_total_mes"] / df["ingreso_mensual"]).round(3)
df["perfil"] = df.apply(etiquetar_con_ambiguedad, axis=1)

print(df["perfil"].value_counts())
df.to_csv("dataset_perfil_financiero.csv", index=False)
df.head()


perfil
En riesgo         433
Saludable         273
En observacion     94
Name: count, dtype: int64


,ingreso_mensual,nivel_endeudamiento,frecuencia_ahorro,gasto_total_mes,ratio_gasto_ingreso,perfil
0,1006605.0,55.5,Baja,401260.0,0.399,En riesgo
1,966843.0,63.6,Media,691147.0,0.715,En riesgo
2,2746887.0,66.1,Baja,1364318.0,0.497,En riesgo
3,973775.0,67.2,Baja,571131.0,0.587,En riesgo
4,1034264.0,36.5,Baja,1129007.0,1.092,En riesgo


## 10. Comparación de modelos candidatos

Con solo 2 variables numéricas y fronteras de decisión simples, vale la pena comparar alternativas antes de asumir que el modelo más "grande" (RandomForest) es el mejor. Se comparan 3 candidatos con validación cruzada (5-fold) y F1-macro (apropiado aquí porque las clases están desbalanceadas):

- **Regresión Logística** — la más simple, asume fronteras lineales
- **Árbol de Decisión** (profundidad 4) — captura fronteras no lineales, totalmente interpretable
- **Random Forest** (100 árboles) — el más complejo, buen desempeño pero "caja negra"


In [13]:
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import f1_score

X = df[["nivel_endeudamiento", "ratio_gasto_ingreso"]]
y = df["perfil"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

candidatos = {
    "Regresion Logistica": LogisticRegression(max_iter=1000, random_state=42),
    "Arbol de Decision (prof. 4)": DecisionTreeClassifier(max_depth=4, random_state=42),
    "Random Forest (100 arboles)": RandomForestClassifier(n_estimators=100, random_state=42),
}

filas = []
for nombre, m in candidatos.items():
    cv = cross_val_score(m, X_train, y_train, cv=5, scoring="f1_macro")
    m.fit(X_train, y_train)
    f1_test = f1_score(y_test, m.predict(X_test), average="macro")
    filas.append({"modelo": nombre, "f1_cv_promedio": round(cv.mean(), 3),
                  "f1_cv_std": round(cv.std(), 3), "f1_test": round(f1_test, 3)})

comparacion = pd.DataFrame(filas)
comparacion


,modelo,f1_cv_promedio,f1_cv_std,f1_test
0,Regresion Logistica,0.554,0.014,0.547
1,Arbol de Decision (prof. 4),0.808,0.024,0.831
2,Random Forest (100 arboles),0.783,0.018,0.848


**Conclusión de la comparación:**

- La Regresión Logística tiene desempeño notablemente inferior (F1 ≈ 0.55) porque la frontera de decisión no es lineal (son dos umbrales independientes que forman una región rectangular, no un plano).
- El Árbol de Decisión y el Random Forest quedan prácticamente empatados (diferencia menor al margen de error de la validación cruzada), pero el árbol usa ~15 hojas contra 100 árboles completos.

**Decisión: se elige el Árbol de Decisión como modelo final.** Con una diferencia de desempeño insignificante frente al Random Forest, el árbol gana por ser totalmente interpretable (se puede mostrar como diagrama de decisión), más liviano, y porque sus cortes aprendidos coinciden casi exactamente con los umbrales de negocio definidos a mano — lo cual sirve como validación cruzada entre las reglas y el modelo.


In [14]:
modelo_final = DecisionTreeClassifier(max_depth=4, random_state=42)
modelo_final.fit(X_train, y_train)

print(classification_report(y_test, modelo_final.predict(X_test)))
print("\nReglas aprendidas por el arbol (comparar con los umbrales de negocio: 36%, 43%, 0.80, 0.90):\n")
print(export_text(modelo_final, feature_names=["nivel_endeudamiento", "ratio_gasto_ingreso"]))


                precision    recall  f1-score   support

En observacion       0.54      0.74      0.62        19
     En riesgo       0.96      0.93      0.95        87
     Saludable       0.96      0.89      0.92        54

      accuracy                           0.89       160
     macro avg       0.82      0.85      0.83       160
  weighted avg       0.91      0.89      0.90       160


Reglas aprendidas por el arbol (comparar con los umbrales de negocio: 36%, 43%, 0.80, 0.90):

|--- nivel_endeudamiento <= 42.20
|   |--- ratio_gasto_ingreso <= 0.90
|   |   |--- nivel_endeudamiento <= 35.75
|   |   |   |--- ratio_gasto_ingreso <= 0.81
|   |   |   |   |--- class: Saludable
|   |   |   |--- ratio_gasto_ingreso >  0.81
|   |   |   |   |--- class: En observacion
|   |   |--- nivel_endeudamiento >  35.75
|   |   |   |--- nivel_endeudamiento <= 41.95
|   |   |   |   |--- class: En observacion
|   |   |   |--- nivel_endeudamiento >  41.95
|   |   |   |   |--- class: Saludable
|   |--- ra

In [15]:
# Verificacion: la probabilidad ahora refleja incertidumbre real
caso_borde = pd.DataFrame([{"nivel_endeudamiento": 37, "ratio_gasto_ingreso": 0.79}])
caso_claro = pd.DataFrame([{"nivel_endeudamiento": 5, "ratio_gasto_ingreso": 0.30}])

for nombre, caso in [("Cerca del borde", caso_borde), ("Caso claro", caso_claro)]:
    pred = modelo_final.predict(caso)[0]
    proba = modelo_final.predict_proba(caso)[0]
    print(f"{nombre}: {caso.to_dict('records')[0]} -> {pred}")
    print(f"  Probabilidades {list(modelo_final.classes_)}: {proba.round(2)}\n")


Cerca del borde: {'nivel_endeudamiento': 37, 'ratio_gasto_ingreso': 0.79} -> En observacion
  Probabilidades ['En observacion', 'En riesgo', 'Saludable']: [0.64 0.14 0.21]

Caso claro: {'nivel_endeudamiento': 5, 'ratio_gasto_ingreso': 0.3} -> Saludable
  Probabilidades ['En observacion', 'En riesgo', 'Saludable']: [0.02 0.   0.98]



In [16]:
joblib.dump(modelo_final, "modelo_perfil_financiero.pkl")
print("Modelo final (Arbol de Decision) guardado: modelo_perfil_financiero.pkl")


Modelo final (Arbol de Decision) guardado: modelo_perfil_financiero.pkl


## 11. Nota de arquitectura: conexión con el backend

Este notebook es solo para exploración, comparación de modelos y entrenamiento — **no se conecta directo al backend**. Lo que sí viaja a producción:

- `modelo_perfil_financiero.pkl` (el árbol entrenado)
- `perfil_financiero.py` — módulo `.py` independiente que carga el modelo una sola vez, combina su predicción (`perfil_financiero` + `probabilidad`) con las reglas de negocio (`razones`), y tiene fallback seguro a reglas puras si el modelo no carga
- `api_perfil.py` — mini-servicio FastAPI opcional, para el caso en que el backend final sea Java y necesite llamar al modelo por HTTP

Ambos archivos ya fueron probados end-to-end (carga del modelo + respuesta HTTP real).
